### Block 0 — Install packages

In [ ]:
!pip install -q openai datasets scikit-learn tqdm pandas


### Block 1 — Imports and Gemini API client

In [ ]:
import os
import getpass
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
from openai import OpenAI


os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Google API Key here: ")


client = OpenAI(
    api_key=os.environ["GOOGLE_API_KEY"],

    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


MODEL_NAME = "gemini-2.5-flash"

print(f"Client configured for model: {MODEL_NAME}")

Paste your Google API Key here: ··········
Client configured for model: gemini-2.5-flash


### Block 2 — Load FIQA dataset

In [ ]:
from huggingface_hub import login
from datasets import load_dataset


login()

In [ ]:
dataset = load_dataset("TheFinAI/flare-fiqasa", split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 3 — Labels and normalization

In [ ]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map the raw model output to one of: 'negative', 'neutral', 'positive'.
    """
    text = (raw or "").strip().lower()

    # Exact match first
    for label in LABELS:
        if label in text:
            return label

    # Handle short variants like "pos" or "neg"
    if "pos" in text:
        return "positive"
    if "neg" in text:
        return "negative"

    # Fallback
    return "neutral"


### Block 4 — Gemini sentiment classifier

In [ ]:
def build_prompt(text: str) -> str:
    """
    Build the instruction prompt for sentiment classification.
    """
    return (
        "You are an expert financial sentiment classifier.\n\n"
        "Classify the sentiment of the following financial news snippet as exactly one of:\n"
        "negative, neutral, positive.\n\n"
        "Return only one word: negative, neutral, or positive.\n\n"
        f"Text: {text}\n"
        "Answer:"
    )


def predict_label(text: str) -> str:
    """
    Call DeepSeek (official API) and return a normalized label.
    """
    prompt = build_prompt(text)

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a financial sentiment classifier. "
                    "Always reply with exactly one word: negative, neutral, or positive."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        max_tokens=8,
        temperature=0.0,
    )

    raw = completion.choices[0].message.content
    return normalize_prediction(raw)


### Block 5 — Evaluation loop over FIQA test split

In [ ]:
y_true = []
y_pred = []

for example in tqdm(dataset, total=len(dataset)):
    text = example["text"]
    true_label = example["answer"].strip().lower()

    y_true.append(true_label)

    pred_label = predict_label(text)
    y_pred.append(pred_label)


100%|██████████| 235/235 [04:02<00:00,  1.03s/it]


### Block 6 — Compute metrics and inspect predictions

In [ ]:
# Overall accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

# Build DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))


Accuracy: 0.0809

Classification report:
              precision    recall  f1-score   support

    negative     0.0000    0.0000    0.0000        76
     neutral     0.0769    1.0000    0.1429        18
    positive     1.0000    0.0071    0.0141       141

    accuracy                         0.0809       235
   macro avg     0.3590    0.3357    0.0523       235
weighted avg     0.6059    0.0809    0.0194       235


First 10 predictions:
                                                                                                                            text     gold    pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative neutral
                                                                         Legal & General share price: Finance chief to step down negative neutral
                                                                 Kingfisher share price slides on cost to implement n

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
